In [ ]:
nn.Module (所有PyTorch模型的基类)
│
├── 参数管理
│   ├── parameters()          # 返回所有可学习参数
│   ├── named_parameters()    # 带名称的参数
│   ├── state_dict()          # 保存模型状态（参数+缓冲区）
│   └── load_state_dict()     # 加载模型状态
│
├── 模块管理
│   ├── modules()             # 返回所有子模块
│   ├── named_modules()       # 带名称的子模块
│   ├── children()            # 直接子模块
│   └── apply()               # 对所有模块应用函数
│
├── 设备管理 (GPU/CPU)
│   ├── to(device)            # 转移到指定设备
│   ├── cuda()                # 转移到GPU
│   ├── cpu()                 # 转移到CPU
│   └── to(dtype)             # 改变数据类型
│
├── 训练状态管理
│   ├── train()               # 设置为训练模式
│   ├── eval()                # 设置为评估模式
│   └── training              # 当前模式标志 (bool)
│
├── 前向传播
│   ├── __call__()            # 调用 forward()
│   ├── forward()             # 子类必须实现
│   └── __setattr__()         # 自动注册子模块和参数
│
└── 钩子机制 (Hook)
    ├── register_forward_pre_hook()
    ├── register_forward_hook()
    └── register_backward_hook()

In [1]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(10, 20)  # weight + bias
        self.linear2 = nn.Linear(20, 5)   # weight + bias
        
model = MyModel()

# 查看所有参数
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    
# 输出:
# linear1.weight: torch.Size([20, 10])
# linear1.bias: torch.Size([20])
# linear2.weight: torch.Size([5, 20])
# linear2.bias: torch.Size([5])

# 获取参数总数
total = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total}")  # 20*10 + 20 + 5*20 + 5 = 325

linear1.weight: torch.Size([20, 10])
linear1.bias: torch.Size([20])
linear2.weight: torch.Size([5, 20])
linear2.bias: torch.Size([5])
总参数量: 325


In [ ]:
# 保存模型
torch.save(model.state_dict(), 'model.pth')

# 加载模型
model = MyModel()
model.load_state_dict(torch.load('model.pth'))

In [ ]:
# 检查GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 方式1: 通用方法
model = model.to(device)
inputs = inputs.to(device)

# 方式2: 快捷方法
model = model.cuda()  # 转GPU
model = model.cpu()   # 转CPU

# 改变数据类型
model = model.to(torch.float16)  # 半精度

In [ ]:
model.train()   # 训练模式
# - Dropout 启用随机丢弃
# - BatchNorm 使用 batch 统计量

model.eval()    # 评估模式
# - Dropout 关闭（使用全部神经元）
# - BatchNorm 使用 running_mean/running_var

# 典型推理流程
model.eval()
with torch.no_grad():  # 不计算梯度，节省内存
    output = model(inputs)

In [3]:
import torch
class Demo(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 以下都会被自动注册：
        
        # 1. 子模块 → 加入 model.modules()
        self.conv = nn.Conv2d(3, 64, 3)
        self.bn = nn.BatchNorm2d(64)
        
        # 2. 参数 → 加入 model.parameters()
        self.weight = nn.Parameter(torch.randn(10, 10))
        
        # 3. 缓冲区 → 加入 model.buffers()（不训练，但保存）
        self.register_buffer('running_mean', torch.zeros(10))
        
        # 4. 普通张量 → 不注册！
        self.temp = torch.randn(5)  # 不会出现在 parameters() 中

demo = Demo()

print("子模块:", list(demo.modules()))
print("参数:", list(demo.parameters()))
print("缓冲区:", list(demo.buffers()))

子模块: [Demo(
  (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1))
  (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
), Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1)), BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)]
参数: [Parameter containing:
tensor([[-0.1482, -0.1865,  0.2722, -1.0413, -0.8914,  0.7255,  0.9278, -1.3159,
          0.1170,  1.8023],
        [-0.4778,  0.6323,  0.1980,  0.2657, -0.0783, -1.9301, -0.4223,  0.1229,
         -1.8044,  0.0896],
        [ 0.3362,  1.6132, -0.2002, -0.4078, -0.2274, -0.5616, -0.4531,  0.3299,
         -0.1123,  1.1115],
        [-0.8969, -0.0739, -0.5792, -1.2523, -1.2397, -0.7355,  1.1992,  0.5953,
          0.3758,  1.0390],
        [ 2.1445,  1.9547,  0.2807, -1.4451,  1.3835, -0.8375, -0.5602, -0.9454,
         -1.4393,  1.2636],
        [ 0.2521, -0.7854, -0.2448, -1.8232, -0.1077,  1.0368,  1.4117,  0.7230,
         -0.4500, -0.2977],
        [-0.4516, -0.6359,

In [ ]:
# ❌ 错误示例
class BadModel(nn.Module):
    def __init__(self):
        # 忘记 super().__init__()
        self.linear = nn.Linear(10, 5)
        
bad = BadModel()
print(list(bad.modules()))      # [] 空！
print(list(bad.parameters()))   # [] 空！
bad.cuda()                      # 报错或无效

# ✅ 正确示例  
class GoodModel(nn.Module):
    def __init__(self):
        super().__init__()  # ← 关键！
        self.linear = nn.Linear(10, 5)
        
good = GoodModel()
print(list(good.modules()))     # [GoodModel, Linear]
print(list(good.parameters()))  # [weight, bias]
good.cuda()                     # 正常工作

In [ ]:
| 功能类别     | 核心方法                           | 用途                   |
| -------- | ------------------------------ | -------------------- |
| **参数管理** | `parameters()`, `state_dict()` | 优化器、保存模型             |
| **设备管理** | `to()`, `cuda()`, `cpu()`      | GPU加速                |
| **训练控制** | `train()`, `eval()`            | 控制 Dropout/BatchNorm |
| **自动注册** | `__setattr__`                  | 自动识别子模块和参数           |
